# Task 1. Dataset and Dependencies

### 1.1. Install and import all required dependencies

In [1]:
%pip install -q pgserver==0.1.4 psycopg2-binary sqlalchemy pandas kagglehub

In [2]:
import pandas as pd
import pgserver
import kagglehub

from pathlib import Path
from sqlalchemy import create_engine, text
from IPython.display import display

Start and check server

In [3]:
pg = pgserver.get_server(
    "/tmp/hw4_pg",
    cleanup_mode="stop"
)

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Add helper function to execute SQL without expected output to show (e.g. create table)

In [4]:
def run_sql(query):
    with engine.begin() as conn:
        conn.exec_driver_sql(query)

Add helper function to execute SQL queries and show output (e.g., SELECT query)

In [5]:
def run_query(query, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(
            text(query),
            conn,
            params=params
        )

Add helper function to execute DML queries (with RETURNING)

In [6]:
def run_dml(query):
    with engine.begin() as conn:
        return pd.read_sql_query(text(query), conn)

### 1.2. Download olist dataset

In [7]:
dataset_dir = Path(
    kagglehub.dataset_download("olistbr/brazilian-ecommerce")
)

print("Dataset location:", dataset_dir)

for path in sorted(dataset_dir.glob("*.csv")):
    print(path.name, f"{path.stat().st_size / 1024**2:.2f} MB")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset location: /kaggle/input/brazilian-ecommerce
olist_customers_dataset.csv 8.62 MB
olist_geolocation_dataset.csv 58.44 MB
olist_order_items_dataset.csv 14.72 MB
olist_order_payments_dataset.csv 5.51 MB
olist_order_reviews_dataset.csv 13.78 MB
olist_orders_dataset.csv 16.84 MB
olist_products_dataset.csv 2.27 MB
olist_sellers_dataset.csv 0.17 MB
product_category_name_translation.csv 0.00 MB


### 1.3. Raw Data Loading

Load data from CSV into raw tables

In [8]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

for table_name, file_name in files.items():
    path = dataset_dir / file_name

    df = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        na_values=[""]
    )

    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
        chunksize=10_000
    )

    print(f"{table_name}: {len(df):,} rows, {len(df.columns)} columns")

olist_customers_raw: 99,441 rows, 5 columns
olist_orders_raw: 99,441 rows, 8 columns
olist_order_items_raw: 112,650 rows, 7 columns
olist_products_raw: 32,951 rows, 9 columns
olist_sellers_raw: 3,095 rows, 4 columns
olist_order_reviews_raw: 99,224 rows, 7 columns


Check number of records in each table

In [9]:
query = """
SELECT 'customers_raw' AS table_name, COUNT(*) AS n_rows
FROM olist_customers_raw
UNION ALL
SELECT 'orders_raw', COUNT(*) FROM olist_orders_raw
UNION ALL
SELECT 'order_items_raw', COUNT(*) FROM olist_order_items_raw
UNION ALL
SELECT 'products_raw', COUNT(*) FROM olist_products_raw
UNION ALL
SELECT 'sellers_raw', COUNT(*) FROM olist_sellers_raw
UNION ALL
SELECT 'order_reviews_raw', COUNT(*) FROM olist_order_reviews_raw
ORDER BY table_name;
"""

display(run_query(query))

,table_name,n_rows
0,customers_raw,99441
1,order_items_raw,112650
2,order_reviews_raw,99224
3,orders_raw,99441
4,products_raw,32951
5,sellers_raw,3095


### 1.4. Typed Schema and Data Conversion

Create typed tables using the schema from learning materials

In [10]:
query = """

DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);

"""

run_sql(query)

Transfer data from raw tables to typed tables (using query from learning materials)

In [11]:
query = """

INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;


"""

run_sql(query)

Run smoke test

In [12]:
query = """

SELECT 'customers' AS table_name, COUNT(*) AS n_rows
FROM olist_customers
UNION ALL
SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL
SELECT 'products', COUNT(*) FROM olist_products
UNION ALL
SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL
SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;

"""

display(run_query(query))

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


# Task 2. PostgreSQL DML Queries

### 2.1. Seller Alert

Create seller alerts table

In [13]:
query = """
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL
        CHECK (alert_type IN (
            'late_delivery', 'low_rating', 'data_quality'
        )),
    severity SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
"""

run_sql(query)

Find seller with the maximum quantity of late orders

In [14]:
query = """
WITH seller_orders AS (
    SELECT DISTINCT
        seller_id,
        order_id
    FROM olist_order_items
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(*) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE
          > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (
    seller_id, alert_type, severity, details
)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders ||
        ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING
    alert_id,
    seller_id,
    alert_type,
    severity,
    created_at;
"""

display(run_dml(query))

,alert_id,seller_id,alert_type,severity,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-10-31 20:19:24.173772+00:00


One late-delivery alert was created for the seller with the highest number of late orders. RETURNING displays the new alert ID and other details, which can be used for audit purposes. Repeating this INSERT alone would create another alert, but the table is recreated during a full notebook run.

### 2.2. Calculate seller ratings without duplicating

Create score table

In [15]:
query = """
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id TEXT PRIMARY KEY
        REFERENCES olist_sellers(seller_id),
    avg_rating NUMERIC(3,2),
    n_reviews INTEGER NOT NULL DEFAULT 0
        CHECK (n_reviews >= 0),
    tier TEXT CHECK (
        tier IN ('gold', 'silver', 'bronze')
    ),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
"""

run_sql(query)

Calculate seller score

In [16]:
query = """
WITH seller_orders AS (
    SELECT DISTINCT
        seller_id,
        order_id
    FROM olist_order_items
),
review_per_order AS (
    SELECT
        order_id,
        AVG(review_score)::NUMERIC(3,2) AS order_review_score
    FROM olist_order_reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3,2) AS avg_rating,
        COUNT(rpo.order_id)::INTEGER AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
),
upserted AS (
    INSERT INTO seller_score (
        seller_id, avg_rating, n_reviews, tier
    )
    SELECT
        seller_id,
        avg_rating,
        n_reviews,
        CASE
            WHEN avg_rating IS NULL THEN NULL
            WHEN avg_rating >= 4.5 THEN 'gold'
            WHEN avg_rating >= 3.5 THEN 'silver'
            ELSE 'bronze'
        END
    FROM seller_rating
    WHERE TRUE
    ON CONFLICT (seller_id) DO UPDATE
    SET
        avg_rating = EXCLUDED.avg_rating,
        n_reviews = EXCLUDED.n_reviews,
        tier = EXCLUDED.tier,
        updated_at = NOW()
    WHERE (
        seller_score.avg_rating,
        seller_score.n_reviews,
        seller_score.tier
    ) IS DISTINCT FROM (
        EXCLUDED.avg_rating,
        EXCLUDED.n_reviews,
        EXCLUDED.tier
    )
    RETURNING seller_id
)
SELECT COUNT(*) AS inserted_or_updated_sellers
FROM upserted;
"""

display(run_dml(query))

,inserted_or_updated_sellers
0,3095


Seller scores were calculated for 3,095 sellers. Order items were first reduced to unique seller-order pairs, and reviews were aggregated by order to avoid counting the same review multiple times. ON CONFLICT allows existing seller scores to be updated without creating duplicates, while the additional condition prevents unnecessary updates.

The review score belongs to the order, not to an individual seller. For orders involving multiple sellers, the same order-level rating is associated with each participating seller.

### 2.3. Update late delivery flag

Add new column to the `olist_orders` table

In [17]:
query = """
ALTER TABLE olist_orders
ADD COLUMN IF NOT EXISTS is_late
BOOLEAN NOT NULL DEFAULT FALSE;
"""

run_sql(query)

Update `is_late` column values

In [18]:
query = """
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE
                > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
),
updated AS (
    UPDATE olist_orders AS o
    SET is_late = f.is_late_calc
    FROM delivery_flags AS f
    WHERE f.order_id = o.order_id
    RETURNING o.order_id, o.is_late
)
SELECT
    COUNT(*) AS updated_orders,
    COUNT(*) FILTER (WHERE is_late) AS late_orders
FROM updated;
"""

display(run_dml(query))

,updated_orders,late_orders
0,99441,6535


The UPDATE query processed 99,441 orders and marked 6,535 as late deliveries. The is_late flag is calculated from actual and estimated delivery dates using UPDATE ... FROM. Repeating the query produces the same flag values when the source dates have not changed.

### 2.4. Remove Old Alerts

Remove seller alerts older than 30 days and display deleted records

In [19]:
query = """
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING
    alert_id,
    seller_id,
    alert_type,
    created_at;
"""

display(run_dml(query))

,alert_id,seller_id,alert_type,created_at


No alerts older than 30 days were found, so the DELETE query did not remove any records. DELETE ... RETURNING would display the details of deleted alerts if matching records existed.

# Task 3: Customer Segmentation

### 3.1. Create Customer Dimension

Create a deduplicated customer table using customer_unique_id as the primary key.

In [20]:
query = """
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT
        REFERENCES olist_customers(customer_id),
    customer_state CHAR(2),
    customer_city TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
"""

run_sql(query)

### 3.2. Create Customer Segments Table

Create a table with customer-level segments and integrity constraints.

In [21]:
query = """
CREATE TABLE customer_segments (
    segment_id BIGINT
        GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id TEXT NOT NULL UNIQUE,

    segment_name TEXT NOT NULL
        CHECK (segment_name IN (
            'VIP', 'regular', 'new', 'inactive'
        )),

    rfm_score SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days INTEGER
        CHECK (recency_days >= 0),

    monetary_value NUMERIC(12,2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

ALTER TABLE customer_segments
ADD CONSTRAINT fk_customer_segments_customer_unique
FOREIGN KEY (customer_unique_id)
REFERENCES olist_customer_dim(customer_unique_id)
ON DELETE CASCADE;
"""

run_sql(query)

### 3.3. Calculate Customer Segments

In [22]:
query = """
WITH reference_date AS (
    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(
            SUM(oi.price + oi.freight_value), 0
        )::NUMERIC(12,2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id)::INTEGER AS n_orders,
        COALESCE(
            SUM(ot.order_total), 0
        )::NUMERIC(12,2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        (rd.as_of_date - cs.last_order_date)::INTEGER
            AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND rd.as_of_date - cs.last_order_date <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2
                 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500 THEN 4
            WHEN cs.n_orders >= 2 THEN 3
            WHEN cs.n_orders = 1 THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
),
upserted AS (
    INSERT INTO customer_segments (
        customer_unique_id,
        segment_name,
        rfm_score,
        recency_days,
        monetary_value,
        n_orders
    )
    SELECT
        customer_unique_id,
        segment_name,
        rfm_score,
        recency_days,
        monetary_value,
        n_orders
    FROM segmented
    WHERE TRUE
    ON CONFLICT (customer_unique_id) DO UPDATE
    SET
        segment_name = EXCLUDED.segment_name,
        rfm_score = EXCLUDED.rfm_score,
        recency_days = EXCLUDED.recency_days,
        monetary_value = EXCLUDED.monetary_value,
        n_orders = EXCLUDED.n_orders,
        assigned_at = NOW()
    WHERE (
        customer_segments.segment_name,
        customer_segments.rfm_score,
        customer_segments.recency_days,
        customer_segments.monetary_value,
        customer_segments.n_orders
    ) IS DISTINCT FROM (
        EXCLUDED.segment_name,
        EXCLUDED.rfm_score,
        EXCLUDED.recency_days,
        EXCLUDED.monetary_value,
        EXCLUDED.n_orders
    )
    RETURNING customer_unique_id
)
SELECT COUNT(*) AS inserted_or_updated_customers
FROM upserted;
"""

display(run_dml(query))

,inserted_or_updated_customers
0,96096


### 3.4. Verification

Check number of customers and their segmentation

In [23]:
query = """
SELECT
    segment_name,
    COUNT(*) AS n_customers,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_customers,
    ROUND(AVG(n_orders), 2) AS avg_orders,
    ROUND(AVG(monetary_value), 2) AS avg_monetary
FROM customer_segments
GROUP BY segment_name
ORDER BY n_customers DESC;
"""

display(run_query(query))

,segment_name,n_customers,pct_customers,avg_orders,avg_monetary
0,regular,45982,47.85,1.07,224.57
1,inactive,40032,41.66,1.00,59.28
2,new,8891,9.25,1.00,140.39
3,VIP,1191,1.24,1.11,1591.81


Check keys and constraints correspondence

In [24]:
query = """
SELECT
    (SELECT COUNT(DISTINCT customer_unique_id)
     FROM olist_customers) AS source_unique_customers,

    (SELECT COUNT(*)
     FROM olist_customer_dim) AS dimension_customers,

    (SELECT COUNT(*)
     FROM customer_segments) AS segmented_customers;
"""

display(run_query(query))

query = """
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
"""

display(run_query(query))

,source_unique_customers,dimension_customers,segmented_customers
0,96096,96096,96096


,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK ((recency_days >= 0))
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


### Customer Segmentation Findings

The customer dimension and segmentation tables both contain 96,096 records, matching the number of unique buyers in the source data.

The largest segments are `regular` (47.85%) and `inactive` (41.66%). The `new` segment represents 9.25% of buyers, while only 1.24% are classified as `VIP`.

Segmentation is based on `customer_unique_id` to combine multiple orders from the same buyer. A separate customer dimension provides the primary key required for the foreign key relationship.

Recency is calculated relative to the latest order date in the historical dataset, making the results reproducible. The `ON CONFLICT` logic allows segments to be recalculated without creating duplicate records.

These segments use simplified educational rules rather than a full RFM scoring model.

# Task 4. JOIN Queries

### 4.1. INNER JOIN: Revenue by Product Category and Seller State

Business Question: Which product categories and seller states generate the highest revenue from delivered orders?

In [25]:
query = """
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
FROM olist_order_items AS oi
INNER JOIN olist_orders AS o
    ON o.order_id = oi.order_id
INNER JOIN olist_products AS p
    ON p.product_id = oi.product_id
INNER JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY
    p.product_category_name,
    s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
"""

display(run_query(query))

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


All top 10 combinations of product category and seller state belong to São Paulo (SP). The highest revenue comes from cama_mesa_banho (`R$1,070,048.14`), followed by relogios_presentes (`R$1,016,168.80`) and beleza_saude (`R$790,967.66`).

The query includes only delivered orders and calculates revenue as item prices plus freight costs. These results show that SP sellers dominate the highest-revenue combinations in this dataset.

### 4.2. LEFT JOIN: Customer-Level Features

Business Question: What customer-level features can be created for each unique buyer, including total spending, number of orders and average review score?

In [26]:
query = """
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        COALESCE(
            SUM(oi.price + oi.freight_value), 0
        )::NUMERIC(12,2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        order_id,
        AVG(review_score)::NUMERIC(3,2)
            AS order_review_score
    FROM olist_order_reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(*) AS n_orders,
        SUM(ot.order_total)::NUMERIC(12,2)
            AS total_spend,
        COUNT(rpo.order_id) AS n_reviewed_orders,
        AVG(rpo.order_review_score)::NUMERIC(3,2)
            AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0) AS total_spend,
    COALESCE(cs.n_reviewed_orders, 0) AS n_reviewed_orders,
    cs.avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC, cd.customer_unique_id
LIMIT 20;
"""

display(run_query(query))

,customer_unique_id,customer_state,n_orders,total_spend,n_reviewed_orders,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,2,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,1,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0,NaN
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,1,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,1,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,1,5.0


The query creates customer-level features using customer_unique_id, including the number of orders, total spending, and average review score. The highest total spending in the displayed results is `R$13,664.08` for a customer with one order and an average review score of 1.00.

Some customers have multiple orders, while others have no available reviews. Aggregating reviews at the order level before joining prevents duplicate review scores, and missing ratings are kept as NULL instead of being replaced with zero.

### 4.3. FULL OUTER JOIN: Customer and Seller States

Business Question: Which Brazilian states have customers, sellers, or both?

In [27]:
query = """
WITH customer_states AS (
    SELECT
        customer_state AS state,
        COUNT(*) AS n_customer_records
    FROM olist_customers
    GROUP BY customer_state
),
seller_states AS (
    SELECT
        seller_state AS state,
        COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    COALESCE(c.n_customer_records, 0) AS n_customer_records,
    COALESCE(s.n_sellers, 0) AS n_sellers,
    CASE
        WHEN c.state IS NOT NULL
         AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
"""

display(run_query(query))

,state,n_customer_records,n_sellers,state_presence
0,AC,81,1,both
1,AL,413,0,customers_only
2,AM,148,1,both
3,AP,68,0,customers_only
4,BA,3380,19,both
5,CE,1336,13,both
6,DF,2140,30,both
7,ES,2033,23,both
8,GO,2020,40,both
9,MA,747,1,both


The FULL OUTER JOIN shows 27 Brazilian states in the combined customer and seller data. Customers and sellers are both present in 23 states, while four states (AL, AP, RR, and TO) have customer records but no sellers.

No states contain sellers without customers. São Paulo (SP) has the highest number of customer records (41,746) and sellers (1,849), showing a strong geographical concentration of the platform's activity.

### 4.4. SELF JOIN: Seller Pairs from the Same State

Business Question: Which pairs of sellers are located in the same state but in different cities?

In [28]:
query = """
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
INNER JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY
    s1.seller_state,
    seller_a,
    seller_b
LIMIT 20;
"""

display(run_query(query))

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


The SELF JOIN identifies pairs of sellers located in the same state but in different cities. The displayed results include seller pairs from Bahia (BA), such as sellers in Lauro de Freitas, Irecê, and Salvador.

The condition s1.seller_id < s2.seller_id prevents self-matches and reversed duplicate pairs. Only the first 20 results are displayed, so this output is a sample rather than a complete list of seller pairs.

### 4.5. Anti-Join: Orders Without Reviews

Business Question: Which orders do not have any associated review records?

In [29]:
query = """
WITH orders_without_reviews AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_status,
        o.order_purchase_timestamp
    FROM olist_orders AS o
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = o.order_id
    WHERE rv.review_row_id IS NULL
)
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    COUNT(*) OVER () AS total_orders_without_reviews
FROM orders_without_reviews
ORDER BY order_purchase_timestamp, order_id
LIMIT 20;
"""

display(run_query(query))

,order_id,customer_id,order_status,order_purchase_timestamp,total_orders_without_reviews
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30,768
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59,768
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22,768
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31,768
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27,768
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41,768
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28,768
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51,768
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36,768
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37,768


The anti-join identified 768 orders without any associated review records, which is approximately 0.77% of all 99,441 orders. The displayed examples include delivered, shipped, and unavailable orders.

The query uses LEFT JOIN ... IS NULL to find orders without matching reviews. The window function COUNT(*) OVER () provides the total number of missing-review orders even though only 20 records are displayed.

# Task 5. GROUP BY, HAVING and Aggregations

### 5.1. Revenue by Product Category

Business Question: Which product categories generate more than `R$10,000` in product sales revenue from delivered orders?

In [30]:
query = """
SELECT
    COALESCE(p.product_category_name, '(unknown)') AS category,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    ROUND(SUM(oi.price), 2) AS product_revenue,
    ROUND(SUM(oi.freight_value), 2) AS freight,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM olist_order_items AS oi
INNER JOIN olist_orders AS o
    ON o.order_id = oi.order_id
INNER JOIN olist_products AS p
    ON p.product_id = oi.product_id
WHERE o.order_status = 'delivered'
GROUP BY COALESCE(p.product_category_name, '(unknown)')
HAVING SUM(oi.price) > 10000
ORDER BY product_revenue DESC;
"""

display(run_query(query))

,category,n_orders,product_revenue,freight,avg_item_price
0,beleza_saude,8647,1233131.72,178957.81,130.28
1,relogios_presentes,5495,1166176.98,98156.14,199.04
2,cama_mesa_banho,9272,1023434.76,201774.50,93.44
3,esporte_lazer,7530,954852.55,163404.36,113.25
4,informatica_acessorios,6530,888724.61,143999.16,116.26
5,moveis_decoracao,6307,711927.69,168402.23,87.25
6,utilidades_domesticas,5743,615628.69,142763.56,90.60
7,cool_stuff,3559,610204.10,81476.79,164.12
8,automotivo,3810,578966.65,90488.10,139.85
9,brinquedos,3804,471286.48,75774.58,116.94


Among delivered orders, 53 product categories have more than `R$10,000` in product revenue. The highest revenue comes from beleza_saude (`R$1,233,131.72`), followed by relogios_presentes (`R$1,166,176.98`) and cama_mesa_banho (`R$1,023,434.76`).

The (unknown) category also meets the threshold, with `R$170,726.63` in product revenue. The WHERE condition filters delivered orders before aggregation, while HAVING selects categories based on their total revenue after aggregation. Freight costs are reported separately and are not included in product revenue.

### 5.2. Average Order Value by Customer State

Business Question: Which customer states have the highest average order value among states with at least 100 delivered orders?

In [31]:
query = """
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12,2)
            AS order_total
    FROM olist_orders AS o
    INNER JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    INNER JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id, c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    ROUND(SUM(order_total), 2) AS total_order_value,
    ROUND(AVG(order_total), 2) AS avg_order_value
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY avg_order_value DESC;
"""

display(run_query(query))

,customer_state,n_orders,total_order_value,avg_order_value
0,PB,517,137838.55,266.61
1,AL,397,94172.49,237.21
2,RO,243,56966.00,234.43
3,PA,946,212023.57,224.13
4,PI,476,105178.19,220.96
5,TO,274,60007.37,219.01
6,RN,474,100714.78,212.48
7,SE,335,70289.13,209.82
8,CE,1279,266436.77,208.32
9,MA,717,147803.55,206.14


Among states with at least 100 delivered orders, Paraíba (PB) has the highest average order value (`R$266.61`), followed by Alagoas (AL) at `R$237.21` and Rondônia (RO) at `R$234.43`.

São Paulo (SP) has the largest number of delivered orders (40,501) and the highest total order value (`R$5,769,703.15`), but its average order value is only `R$142.46`.

The query first calculates the total value of each order, including freight, and then calculates the average by customer state. This prevents the average item price from being incorrectly used as the average order value.

### 5.3. Product Categories of Repeat Customers

Business Question: Which product categories have repeat customers purchased, and which repeat customers have the highest total product spending?

In [32]:
query = """
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    ROUND(SUM(oi.price), 2) AS total_product_spend,
    STRING_AGG(
        DISTINCT COALESCE(
            p.product_category_name, '(unknown)'
        ),
        ', '
        ORDER BY COALESCE(
            p.product_category_name, '(unknown)'
        )
    ) AS purchased_categories
FROM olist_customers AS c
INNER JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
INNER JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_product_spend DESC
LIMIT 20;
"""

display(run_query(query))

,customer_unique_id,n_orders,total_product_spend,purchased_categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
2,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
3,d77aa95864ae5b42160937615628723a,2,2400.00,pcs
4,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
5,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
6,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
7,73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games
8,1da09dd64e235e7c2f29a4faff33535c,3,1980.28,informatica_acessorios
9,906a8a4ec9f3d4c3e64fa6d1c4fe6009,2,1835.00,"agro_industria_e_comercio, utilidades_domesticas"


Among repeat buyers with at least two delivered orders, the highest product spending in the displayed results is `R$7,388.00` for a buyer with two orders in the eletroportateis category. The second-highest amount is `R$4,080.00` across four orders in telefonia.

The STRING_AGG(DISTINCT ...) function combines unique product categories purchased by each customer. The HAVING condition selects buyers with at least two delivered orders.

Only the top 20 buyers by product spending are displayed, and freight costs are not included in this metric.

# Task 6. Set Operations, Window Functions and Reflection

### 6.1. Set Operation: Common Customer and Seller States

Business Question: Which Brazilian states have both customers and sellers in the Olist dataset?

In [33]:
query = """
SELECT
    customer_state AS state
FROM olist_customers

INTERSECT

SELECT
    seller_state AS state
FROM olist_sellers

ORDER BY state;
"""

display(run_query(query))

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


The query found 23 Brazilian states where both customers and sellers are present. This matches the results of the FULL OUTER JOIN in Task 4.3, which also identified four states with customers but no sellers.

INTERSECT was used because we need only the states present in both datasets. Unlike UNION, it excludes states found in only one dataset and returns each matching state once.

### 6.2. Window Function: Latest Order per Unique Customer

Business Question: What is the latest order for each unique customer, based on the order purchase timestamp?

In [34]:
query = """
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        o.order_status,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY
                o.order_purchase_timestamp DESC,
                o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    INNER JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp,
    order_status
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
"""

display(run_query(query))

,customer_unique_id,customer_id,order_id,order_purchase_timestamp,order_status
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,delivered
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,delivered
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,delivered
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,delivered
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,delivered
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16,delivered
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23,delivered
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41,delivered
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12,delivered
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12,delivered


The query uses ROW_NUMBER() to identify the latest order for each unique customer. Orders are ranked by purchase timestamp within each customer_unique_id, and only the record with rank 1 is selected.

The additional ordering by order_id makes the result deterministic when purchase timestamps are equal. Only 20 records are displayed, but the ranking is calculated across all orders. This approach is useful when the complete latest order is needed, rather than just the maximum timestamp.

### 6.3. Reflection

In this homework, I worked with six related Olist tables and built a typed PostgreSQL schema with primary keys, foreign keys, checks and a customer segmentation table. The most difficult JOIN for me was FULL OUTER JOIN, because I had to keep records that exist only on one side and correctly identify the missing matches. The SELF JOIN was the hardest to turn into a useful business question; comparing sellers in the same state but different cities was a reasonable example.

For production ETL, ON CONFLICT seems the most useful PostgreSQL feature. It allows seller scores and customer segments to be recalculated without creating duplicate records. RETURNING is also helpful to check what was inserted, updated or deleted.

In a real ML project, I would use window functions to find the latest purchase, rank orders and calculate customer activity over time. The Olist data also contains quality issues: 768 orders have no review, and some products have no category. These missing values should be kept visible rather than replaced with misleading values.

For customer-level features I used customer_unique_id instead of customer_id, since one buyer may have several order-specific IDs. Finally, seller ratings require special care: order items are reduced to unique seller-order pairs and reviews are aggregated by order. Otherwise, joining item-level rows directly to reviews could count the same review multiple times. An order-level review can still apply to several sellers, which is a limitation of this dataset.